# L10a Advanced: The Leisen–Reimer Tree and the Greeks of American Options
The [lecture](../../CHEME-5660-L10a-Lecture-American-Derivatives-Greeks-Fall-2026.ipynb) estimates each Greek by repricing the CRR tree with one input moved up and down. Why can this give a Gamma of zero, and how does the Leisen–Reimer tree avoid it?

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __Explain why repricing the CRR tree fails:__ Describe the CRR premium as straight pieces joined at corners where a terminal node crosses the strike. Explain why a central difference between two corners gives a Gamma of zero and a Delta that jumps.
> * __Build the Leisen–Reimer tree:__ Derive the risk-neutral probability that geometric Brownian motion finishes above the strike. Use the Peizer–Pratt inversion to set the up probability and the factors so the strike stays between the middle terminal nodes.
> * __Compute the Greeks on the Leisen–Reimer tree:__ Estimate Delta, Gamma, Theta, Vega, and Rho by central differences that rebuild the tree for each input. Check that each Greek stops changing with more tree steps and smaller differencing steps.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L10a) lists these symbols with their meanings and units.

Let's get started!
___

## Repricing the CRR Tree
The lecture reprices the premium $\mathcal{P}_{\star}$ of a contract of type $\star\in\left\{c,p\right\}$ with one input moved up and down by a step $\epsilon$:
$$
\begin{align*}
\frac{\partial\mathcal{P}_{\star}}{\partial x} & \approx \frac{\mathcal{P}_{\star}(x+\epsilon)-\mathcal{P}_{\star}(x-\epsilon)}{2\epsilon}\qquad x\in\left\{S,\sigma,t,g_{y}\right\}\\
\Gamma_{\star} & \approx \frac{\mathcal{P}_{\star}(S+\epsilon)-2\,\mathcal{P}_{\star}(S)+\mathcal{P}_{\star}(S-\epsilon)}{\epsilon^{2}}
\end{align*}
$$

__What goes wrong on the CRR tree?__ A terminal node's share price is $S_{0}u^{k}d^{h-k}$ after $k$ up moves by the factor $u$ and $h-k$ down moves by $d$. Its payoff is zero until that price passes the strike, then rises in a straight line.

The CRR factors do not depend on $S_{0}$, so the premium adds these bent lines with fixed weights. The result is straight pieces joined at corners where a node crosses the strike (top panel):

<div>
    <center>
        <img src="../../figs/Fig-L10a-CRR-vs-LR-Delta.svg" width="700" alt="Two stacked panels against the share price today, from 90 to 110 USD/share, for a call with strike 100 USD/share on trees with 9 steps. Top, the premium: the CRR premium is straight pieces joined at two corners, near 95 and 105 USD/share, where a terminal node crosses the strike. A dashed line extends the middle piece, and the CRR premium leaves it at each corner. Three open circles at S0 minus epsilon, S0, and S0 plus epsilon lie on that one piece, so the central difference gives a Gamma of zero. The Leisen–Reimer premium is a smooth curve. Bottom, Delta, the slope of the premium: the CRR Delta is flat on each piece and jumps at each corner, while the Leisen–Reimer Delta rises smoothly."/>
    </center>
</div>

A step $\epsilon$ between two corners sees one straight piece (open circles), so Delta is that piece's slope and Gamma is zero. The CRR Delta is flat on each piece and jumps at each corner (bottom panel). More tree steps move the corners closer but never remove them. Leisen and Reimer remove them with a tree built around the strike.

__Reference:__
* [Leisen, D. P. J.; Reimer, M. (1996). "Binomial models for option valuation – examining and improving convergence". Applied Mathematical Finance. 3 (4): 319–346. doi:10.1080/13504869600000015](https://doi.org/10.1080/13504869600000015)
___

## Building the Leisen–Reimer Tree
Leisen and Reimer choose the up probability $q$ so the tree finishes above the strike as often as geometric Brownian motion does under $\mathbb{Q}$. That probability depends only on the share price, so it holds for American contracts too.

Under $\mathbb{Q}$, with volatility $\sigma$, risk-free rate $g_{y}$, and time to expiration $T$ (years), the share price at expiration and its logarithm are:
$$
\begin{align*}
S_{T} & = S_{0}\exp\left[\left(g_{y}-\frac{\sigma^{2}}{2}\right)T+\sigma\sqrt{T}\,Z\right] && \text{GBM under }\mathbb{Q}\text{, with }Z\sim\mathcal{N}(0,1)\\
\ln S_{T} & = \ln S_{0}+\left(g_{y}-\frac{\sigma^{2}}{2}\right)T+\sigma\sqrt{T}\,Z && \text{take the logarithm}
\end{align*}
$$
Then the probability that the share price finishes above the strike $K$ is:
$$
\begin{align*}
\mathbb{Q}\left(S_{T}>K\right) & = \mathbb{Q}\left(\ln S_{T}>\ln K\right) && \ln\text{ is increasing}\\
& = \mathbb{Q}\left(\ln S_{0}+\left(g_{y}-\frac{\sigma^{2}}{2}\right)T+\sigma\sqrt{T}\,Z>\ln K\right) && \text{substitute }\ln S_{T}\\
& = \mathbb{Q}\left(Z>\frac{\ln K-\ln S_{0}-\left(g_{y}-\sigma^{2}/2\right)T}{\sigma\sqrt{T}}\right) && \text{subtract, then divide by }\sigma\sqrt{T}>0\\
& = \mathbb{Q}\left(Z>-d_{-}\right) && \ln K-\ln S_{0}=-\ln\left(S_{0}/K\right)\\
& = 1-N(-d_{-}) && N\text{ is the standard normal CDF}\\
& = N(d_{-}) && N(-z)=1-N(z)\text{, by symmetry about zero}
\end{align*}
$$
where $d_{-}=\left[\ln(S_{0}/K)+\left(g_{y}-\sigma^{2}/2\right)T\right]/(\sigma\sqrt{T})$.

Leisen and Reimer use an odd number of steps $h$ and center the tree on the strike, so it finishes above the strike when more than half of its moves are up. The Peizer–Pratt inversion $H(z;h)$ gives the up probability that makes this happen with probability about $N(z)$.

A second probability $q'$, built from $d_{+}=d_{-}+\sigma\sqrt{T}$, sets the factors:
$$
\begin{align*}
H(z;h) & = \frac{1}{2}+\frac{\operatorname{sgn}(z)}{2}\sqrt{1-\exp\left[-\left(\frac{z}{h+1/3+0.1/(h+1)}\right)^{2}\left(h+\frac{1}{6}\right)\right]} && \text{Peizer–Pratt inversion}\\
q & = H(d_{-};h) && \text{risk-neutral up probability}\\
q' & = H(d_{+};h) && \text{second probability}\\
u & = e^{g_{y}\Delta t}\,\frac{q'}{q} && \text{up factor}\\
d & = e^{g_{y}\Delta t}\,\frac{1-q'}{1-q} && \text{down factor}
\end{align*}
$$
where $\Delta t=T/h$.

Like the CRR factors, they grow the expected share price at the risk-free rate over one step:
$$
\begin{align*}
qu+(1-q)d & = q\,e^{g_{y}\Delta t}\frac{q'}{q}+(1-q)\,e^{g_{y}\Delta t}\frac{1-q'}{1-q} && \text{substitute } u \text{ and } d\\
& = e^{g_{y}\Delta t}\left(q'+1-q'\right) && \text{cancel } q \text{ and } 1-q\\
& = e^{g_{y}\Delta t} && q'+(1-q')=1
\end{align*}
$$
Because $d_{\pm}$ depend on $S_{0}$, so do the factors. The strike stays between the two middle terminal nodes as $S_{0}$ moves, so the premium has no corners. Backward induction and the exercise decision work as on the CRR tree.

In the lecture, a call's Delta on the CRR tree is a probability of exercise computed with the up probability $q^{*}=q\,u\,e^{-g_{y}\Delta t}$.

On the Leisen–Reimer tree, $q^{*}$ is the second probability $q'$:
$$
\begin{align*}
q^{*} & = q\,u\,e^{-g_{y}\Delta t} && \text{definition of } q^{*}\\
& = q\,e^{g_{y}\Delta t}\,\frac{q'}{q}\,e^{-g_{y}\Delta t} && \text{substitute } u\\
& = q' && \text{cancel } q \text{ and } e^{g_{y}\Delta t}
\end{align*}
$$
Next, we use this tree to compute the Greeks.
___

## Computing the Greeks on the Leisen–Reimer Tree
Let's outline pseudocode for the Greeks of a contract on the Leisen–Reimer tree.

__Initialize__: Given the share price $S_{0}$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$ (units: $\mathrm{yr}^{-1/2}$), the risk-free rate $g_{y}$ (units: $\mathrm{yr}^{-1}$), an odd number of time steps $h$, and a step $\epsilon_{x}$ for each input $x\in\left\{S_{0},\sigma,T,g_{y}\right\}$. Let $\mathcal{P}(S_{0},\sigma,T,g_{y})$ be the premium that steps 1–4 compute.

1. Compute the time step $\Delta t \gets T/h$, then $d_{-} \gets \left[\ln(S_{0}/K)+\left(g_{y}-\sigma^{2}/2\right)T\right]/(\sigma\sqrt{T})$ and $d_{+} \gets d_{-}+\sigma\sqrt{T}$.
2. Compute the probabilities $q \gets H(d_{-};h)$ and $q' \gets H(d_{+};h)$.
3. Compute the factors $u \gets e^{g_{y}\Delta t}\,q'/q$ and $d \gets e^{g_{y}\Delta t}\,(1-q')/(1-q)$.
4. Compute the premium with the CRR pricing pseudocode, using these $u$, $d$, and $q$: $\mathcal{P}(S_{0},\sigma,T,g_{y}) \gets \hat{V}_{0,0}$.
5. For each input $x\in\left\{S_{0},\sigma,T,g_{y}\right\}$ __do__:
   - a. Compute $\mathcal{P}(x+\epsilon_{x})$ and $\mathcal{P}(x-\epsilon_{x})$ with steps 1–4, moving only $x$. The factors depend on every input, so each evaluation rebuilds the tree.
   - b. Compute the central difference: $\partial\mathcal{P}/\partial x \gets \left[\mathcal{P}(x+\epsilon_{x})-\mathcal{P}(x-\epsilon_{x})\right]/(2\epsilon_{x})$.
6. Compute the first-order Greeks: $\Delta \gets \partial\mathcal{P}/\partial S_{0}$, $\nu \gets \partial\mathcal{P}/\partial\sigma$, $\rho \gets \partial\mathcal{P}/\partial g_{y}$, and $\Theta \gets -\partial\mathcal{P}/\partial T$, because calendar time moves forward as the time to expiration falls.
7. Compute Gamma from the three share price evaluations: $\Gamma \gets \left[\mathcal{P}(S_{0}+\epsilon_{S_{0}})-2\,\mathcal{P}(S_{0})+\mathcal{P}(S_{0}-\epsilon_{S_{0}})\right]/\epsilon_{S_{0}}^{2}$.

__Output__: The premium $\mathcal{P}(S_{0},\sigma,T,g_{y})$ and the Greeks $\Delta$, $\Gamma$, $\Theta$, $\nu$, and $\rho$.

Before reporting a Greek, recompute it with about twice as many time steps, still odd, and smaller steps $\epsilon_{x}$. Report it once it stops changing.
___

## Summary
We showed why central differences on the CRR tree can give a Gamma of zero, and built the Leisen–Reimer tree, whose premium has no corners. We then wrote pseudocode that computes the Greeks by rebuilding that tree for each moved input.

> __Key Takeaways:__
>
> * __The CRR premium has corners:__ We saw that the CRR factors do not depend on the share price today, so the premium is straight pieces joined where a terminal node crosses the strike. A step between two corners gives a Gamma of zero, and more tree steps never remove the corners.
> * __The Leisen–Reimer tree is built around the strike:__ We derived the risk-neutral probability that the share price finishes above the strike and used the Peizer–Pratt inversion to choose an up probability that matches it. The factors then depend on the share price today, so the strike stays between the middle terminal nodes and the corners disappear.
> * __Each Greek rebuilds the tree:__ We repriced the contract with one input moved up and down, rebuilding the tree each time because its factors depend on every input. We report a Greek once it stops changing with more tree steps and smaller differencing steps.

Return to the [lecture](../../CHEME-5660-L10a-Lecture-American-Derivatives-Greeks-Fall-2026.ipynb), or explore the [Greeks example](../../CHEME-5660-L10a-Example-Greeks-PutCall-Fall-2026.ipynb), which computes call and put Greeks on this tree and compares them with the archive's quotes.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___